[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Elheims/scikit-learn-cookbook/blob/main/01.%20API%20Elements%20of%20scikit-learn/ch01_sklearn_api.ipynb)

# Chapter 1: Common Conventions and API Elements of scikit-learn

**Book:** *scikit-learn Cookbook, Third Edition* (John Sukup, Packt) — Chapter 1, pp. 24–35

**What you will learn in this chapter:**
- The design philosophy behind scikit-learn's unified API
- How estimators implement `fit()` and `predict()` for supervised and unsupervised learning
- How transformers use `fit()` and `transform()` to preprocess data
- How to build custom estimators and transformers
- How pipelines chain multiple steps into a single reproducible workflow
- How to access model attributes (`coef_`, `intercept_`) and evaluation methods (`score()`)
- How to tune hyperparameters with `set_params()`, `get_params()`, `GridSearchCV()` and `RandomizedSearchCV()`
- How tags and metadata routing work in scikit-learn

## Chapter summary
Chapter 1 is the conceptual foundation for the whole book. It explains the *design philosophy* of scikit-learn
(consistency, simplicity, modularity, reusability) and the handful of API objects every later chapter relies on:

| Section | Core idea | Key API |
|---|---|---|
| Design philosophy | One uniform interface for all algorithms | `fit`, `predict`, `transform` |
| Estimators | Objects that *learn from data* | `fit`, `predict`, `fit_predict` |
| Transformers | Objects that *change data* using learned parameters | `fit`, `transform`, `fit_transform` |
| Custom estimators/transformers | Extend the library by subclassing | `BaseEstimator`, mixins |
| Pipelines | Chain steps into one object | `Pipeline` |
| Common attributes & methods | Inspect what a model learned | `coef_`, `intercept_`, `score` |
| Hyperparameter tuning | Search the configuration space | `GridSearchCV`, `RandomizedSearchCV`, `get_params/set_params` |
| Metadata: tags and routing | Estimators declare capabilities; metadata flows between steps | tags, metadata routing |
| Best practices | Habits that keep code reproducible | pipelines, CV, uniform API |

**How this notebook is organised.** Each section has (1) a short summary, (2) the book's code reproduced and executed,
and (3) a *Theory* cell. Cells marked **Extension** are not in the book's code (the book only describes the idea);
I added them so the concept can actually be seen working.

In [43]:
import numpy as np
import sklearn
print("scikit-learn version:", sklearn.__version__)
print("numpy version:", np.__version__)

scikit-learn version: 1.9.1
numpy version: 2.5.3


## 1. Introduction to scikit-learn's design philosophy

**Summary.** scikit-learn is built on four principles — *consistency, simplicity, modularity, reusability*. Almost every
object exposes the same verbs: `fit()` to learn, `predict()` to predict, `transform()` to change data. Because of this,
swapping one model for another is usually a one-line change. Components (estimators, transformers, pipelines) can be
combined like Lego bricks.

> The name is always lowercase: **scikit-learn**, pronounced *sy-kit*, "science kit".

> **A little history (from the book's introduction).** The project started in 2007 and officially launched in 2009. Despite its age it remains one of the most widely used libraries in research, academia
> and production applications at scale — and often the first library people meet when they start ML programming.

### Theory
* **Uniform interface (duck typing).** scikit-learn does not force a base class; any object with the right methods works.
  This is why 3rd-party libraries (XGBoost, LightGBM, imbalanced-learn) plug into `Pipeline` and `GridSearchCV`.
* **Separation of learning and applying.** `fit` stores *state* learned from data in attributes ending in an underscore
  (e.g. `coef_`); `predict`/`transform` apply that state without changing it. This is what makes train/test separation
  and reproducibility possible.
* **Composition over inheritance.** Complex workflows are built by *composing* simple objects (a `Pipeline` of
  transformers + one final estimator) rather than by writing one giant class.
* **Sensible defaults + explicit hyperparameters.** Hyperparameters are constructor arguments; learned values are
  trailing-underscore attributes. This convention is used everywhere.

**Demonstration (Extension):** the same three lines work for completely different algorithms — that *is* the design
philosophy. Only the class name changes.

In [44]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

X_iris, y_iris = load_iris(return_X_y=True)
for Model in (LogisticRegression, DecisionTreeClassifier, KNeighborsClassifier):
    m = Model(max_iter=1000) if Model is LogisticRegression else Model()   # 1. construct (set hyperparameters)
    m.fit(X_iris, y_iris)       # 2. learn
    print(f"{Model.__name__:24s} accuracy on training data = {m.score(X_iris, y_iris):.3f}")

LogisticRegression       accuracy on training data = 0.973
DecisionTreeClassifier   accuracy on training data = 1.000
KNeighborsClassifier     accuracy on training data = 0.967


## 2. Understanding estimators

**Summary.** An *estimator* is any object that learns from data. Its two essential methods are `fit()` (learn from
training data) and `predict()` (predict for new data). `fit_predict()` is a shortcut that does both on the *same* data —
typical for unsupervised learning such as clustering, where there is no target to hold out.

### Reproduction of the book's code — `LinearRegression`

In [45]:
from sklearn.linear_model import LinearRegression
import numpy as np

# Example data
X = np.array([[1], [2], [3], [4], [5]])  # Feature matrix
y = np.array([1, 2, 3, 3.5, 5])  # Target values

# Create and fit the model
model = LinearRegression()
model.fit(X, y)

# Predict values for new data
X_new = np.array([[6], [7]])
predictions = model.predict(X_new)
print(predictions)

[5.75 6.7 ]


> **Note on the book's output.** The book prints `[5.75, 6.7]` (a plain list). Real scikit-learn returns a NumPy array,
> so you see `[5.75 6.7 ]` without commas. The values are identical.

### Reproduction — `fit_predict()` with `KMeans`

In [46]:
# Fit_predict is not used in LinearRegression,
# but as an example for clustering:
from sklearn.cluster import KMeans

# Example data
X = np.array([[1], [2], [3], [4], [5]])

# KMeans Clustering example
kmeans = KMeans(n_clusters=2)
labels = kmeans.fit_predict(X)
print(labels)

[1 1 0 0 0]


> **Note.** The book shows `[0,0,0,1,1]` and this run matches. Cluster *numbers* are arbitrary labels, so another run
> may print `[1 1 1 0 0]`, and the middle point (3) sits between the two centres and can land in either group.
> Without `random_state`/`n_init` set, results can differ between runs and versions. Below I make it deterministic.

In [47]:
kmeans = KMeans(n_clusters=2, n_init=10, random_state=42)
print(kmeans.fit_predict(X))
print("Cluster centres:", kmeans.cluster_centers_.ravel())

[0 0 0 1 1]
Cluster centres: [2.  4.5]


### Theory
* **Estimator contract.** `fit(X, y=None)` returns `self` (so calls can be chained: `Model().fit(X, y).predict(X2)`).
* **Supervised vs unsupervised.** Supervised estimators need `y`; unsupervised ones ignore it (`y=None`).
* **Linear regression** models $\hat y = w x + b$ and finds $w, b$ minimising the sum of squared errors
  $\sum_i (y_i - \hat y_i)^2$ (ordinary least squares). For our data the learned line is
  $\hat y = 0.95x + 0.05$, so $x=6 \Rightarrow 5.75$ and $x=7 \Rightarrow 6.70$.
* **K-Means** alternates two steps: assign each point to its nearest centre, then move each centre to the mean of its
  points; it minimises within-cluster variance (*inertia*). It is initialised randomly, hence `random_state`.
* **Why separate `fit` and `predict`?** In supervised learning you must be able to learn on training data and then
  predict on *unseen* data — one fused call cannot do that. `fit_predict` only makes sense when you want labels for the
  data you fit on (clustering).

## 3. Transformers and the `transform()` method

**Summary.** A *transformer* modifies data (scale, encode, impute…). `fit()` **learns** parameters from the data (e.g. mean
and standard deviation), `transform()` **applies** them. `fit_transform()` does both in one call and is used on the
*training* data only; the *test* data must only be passed through `transform()`.

### Reproduction — `StandardScaler`

In [48]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# Example data
X = np.array([[1, 2], [3, 4], [5, 6]])

# Create a StandardScaler instance
scaler = StandardScaler()

# Fit the scaler on the data
scaler.fit(X)

# Transform the data
X_scaled = scaler.transform(X)
print(X_scaled)

[[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]


In [49]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# Example data
X = np.array([[1, 2], [3, 4], [5, 6]])

# Create a StandardScaler instance and
# fit_transform the data in one step
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(X_scaled)

[[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]


**Verify by hand:** for column 1 the mean is 3 and the population std is $\sqrt{8/3}\approx1.633$, so
$(1-3)/1.633 = -1.2247$. The learned statistics live in `scaler.mean_` and `scaler.scale_`.

In [50]:
print("mean_ :", scaler.mean_)
print("scale_:", scaler.scale_)
print("manual:", (X - X.mean(axis=0)) / X.std(axis=0))   # np.std uses ddof=0, same as StandardScaler

mean_ : [3. 4.]
scale_: [1.63299316 1.63299316]
manual: [[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]


**Extension — why you must not `fit` on the test set (data leakage).** The book explains this in words; here it is in numbers.

In [51]:
X_train = np.array([[1.0], [2.0], [3.0], [4.0]])
X_test  = np.array([[10.0]])             # a test point that is far from the training data

good = StandardScaler().fit(X_train)     # learn ONLY from training data
print("Correct  (train statistics)   :", good.transform(X_test).ravel())

bad = StandardScaler().fit(X_test)       # wrong: fitting on the test data
print("Wrong    (fit on test set)    :", bad.transform(X_test).ravel(), " <- always 0, information is destroyed")

Correct  (train statistics)   : [6.70820393]
Wrong    (fit on test set)    : [0.]  <- always 0, information is destroyed


### Theory
* **Standardisation (z-score):** $z = \dfrac{x-\mu}{\sigma}$. After the transform each feature has mean 0 and unit variance.
  Many models (SVM, KNN, linear/logistic regression with regularisation, PCA, neural nets) are sensitive to feature scale.
* **Fit on train, transform on test.** $\mu,\sigma$ are *parameters of the model*. Re-estimating them on the test set
  would (a) leak information from the test set and (b) make train and test live in different coordinate systems.
  The same applies to imputers, encoders, PCA, etc.
* `fit_transform(X)` is equivalent to `fit(X).transform(X)` but may be implemented more efficiently.
* The book defers other transformers (`MinMaxScaler`, `OneHotEncoder`) to **Chapter 2**.

## 4. Handling custom estimators and transformers

**Summary.** scikit-learn is extensible: subclass `BaseEstimator` plus the right *mixin* (`TransformerMixin`,
`RegressorMixin`, `ClassifierMixin`…) and implement `fit` and `transform`/`predict`. The custom object then works inside
`Pipeline`, `GridSearchCV`, cross-validation, etc. The book only *describes* this (no code); full treatment is promised later.

### Theory
* **Mixin** = a small class that adds a piece of behaviour through (multiple) inheritance without being a "parent" in the
  conceptual sense. `TransformerMixin` gives you `fit_transform()` for free if you define `fit` and `transform`.
  `RegressorMixin`/`ClassifierMixin` give a default `score()` (R² / accuracy).
* `BaseEstimator` gives `get_params()` / `set_params()` (by inspecting the `__init__` signature) and a nice `repr`.
  Rule: `__init__` must only *store* hyperparameters as attributes of the same name — no logic, no data.
* `check_is_fitted()` raises `NotFittedError` if `predict/transform` is called before `fit` (it looks for attributes
  ending with `_`).

**Extension — a minimal custom transformer and custom classifier** (illustrating the rules above).

In [52]:
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.utils.validation import check_is_fitted, validate_data

class LogTransformer(TransformerMixin, BaseEstimator):
    """Apply log(1 + x) to every feature (useful for right-skewed data)."""
    def __init__(self, offset=1.0):
        self.offset = offset                       # only store hyperparameters

    def fit(self, X, y=None):
        X = validate_data(self, X)                 # checks input & records n_features_in_
        self.min_seen_ = X.min()                   # a learned attribute ends with "_"
        return self                                # fit must return self

    def transform(self, X):
        check_is_fitted(self)                      # fail if fit() was not called
        X = validate_data(self, X, reset=False)
        return np.log(X + self.offset)

class MajorityClassifier(ClassifierMixin, BaseEstimator):
    """Always predicts the most frequent class seen in training (a baseline)."""
    def fit(self, X, y):
        X, y = validate_data(self, X, y)
        values, counts = np.unique(y, return_counts=True)
        self.classes_ = values
        self.majority_ = values[np.argmax(counts)]
        return self
    def predict(self, X):
        check_is_fitted(self)
        X = validate_data(self, X, reset=False)
        return np.full(X.shape[0], self.majority_)

lt = LogTransformer()
print(lt.fit_transform(np.array([[0.0], [1.0], [9.0]])).ravel())     # fit_transform came free from TransformerMixin
print(lt)                                                             # repr came free from BaseEstimator
print(lt.get_params())

clf = MajorityClassifier().fit(X_iris, y_iris)
print("MajorityClassifier accuracy (score() came free from ClassifierMixin):", round(clf.score(X_iris, y_iris), 3))

try:
    LogTransformer().transform(np.array([[1.0]]))
except Exception as e:
    print("Calling transform before fit ->", type(e).__name__)

[0.         0.69314718 2.30258509]
LogTransformer()
{'offset': 1.0}
MajorityClassifier accuracy (score() came free from ClassifierMixin): 0.333
Calling transform before fit -> NotFittedError


## 5. Pipelines and workflow automation

**Summary.** A `Pipeline` chains transformers and a final estimator into a *single* estimator. Calling
`pipeline.fit(X_train, y_train)` runs `fit_transform` on every intermediate step and `fit` on the last step;
`pipeline.predict(X_test)` runs `transform` on every step and `predict` at the end. This guarantees the same
preprocessing is applied to train and test data. The book also introduces **MLOps** (automating development, testing,
deployment and monitoring of ML models) and states that pipelines, `GridSearchCV` and `joblib`/`pickle` persistence
support it. Detailed pipeline code appears in the book's later chapters (the text points forward to later chapters).

**Extension — a pipeline in action** (with leakage-free cross-validation).

In [53]:
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X_tr, X_te, y_tr, y_te = train_test_split(X_iris, y_iris, test_size=0.3, random_state=42, stratify=y_iris)

pipe = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=200)),
])
pipe.fit(X_tr, y_tr)                      # scaler.fit_transform(X_tr) -> clf.fit
print("Test accuracy:", round(pipe.score(X_te, y_te), 3))   # scaler.transform(X_te) -> clf.predict

# make_pipeline names the steps automatically
print(make_pipeline(StandardScaler(), LogisticRegression()).named_steps.keys())

# In cross-validation the scaler is re-fitted inside each fold -> no leakage
print("5-fold CV accuracy:", cross_val_score(pipe, X_iris, y_iris, cv=5).round(3))

Test accuracy: 0.911
dict_keys(['standardscaler', 'logisticregression'])
5-fold CV accuracy: [0.967 1.    0.933 0.9   1.   ]


Pipelines also render as a diagram in a notebook (this is the *visualising pipelines* feature the book uses in Chapter 2):

In [54]:
pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](3,)","[0,1,2]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,4
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


(This interactive diagram plays the role of the book's **Figure 1.1**, which sketches data flowing through the transformation steps of a `Pipeline()`: each box is a step, and clicking it shows its parameters.)

**Extension — persistence (the "deploy" part of MLOps):** a fitted pipeline is one object, so it can be saved and
loaded with `joblib`.

In [55]:
import joblib, tempfile, os
path = os.path.join(tempfile.gettempdir(), "iris_pipeline.joblib")
joblib.dump(pipe, path)
restored = joblib.load(path)
print("Restored pipeline gives identical predictions:", (restored.predict(X_te) == pipe.predict(X_te)).all())
os.remove(path)

Restored pipeline gives identical predictions: True


### Theory
* **Why pipelines?** (1) *Convenience* — one `fit`/`predict`; (2) *No leakage* — transformers are fit only on the training
  portion of each CV fold; (3) *Joint tuning* — hyperparameters of every step can be searched together using the
  `stepname__param` syntax; (4) *Deployment* — a single artefact to serialise.
* **MLOps.** The ML life cycle is a loop (train → deploy → monitor → retrain). MLOps applies DevOps ideas (automation,
  CI/CD, reproducibility, monitoring) to that loop to handle problems such as *model drift* (data or relationships change
  over time) and *technical debt*. scikit-learn does not do MLOps by itself, but its pipelines/persistence make models easy
  to hand to tools like MLflow or Kubeflow. The book gives **four reasons** MLOps is essential in production:
  1. it **bridges the gap** between data science, ML engineering and operations teams ("less of a *this is your job, this is our job* mindset");
  2. it **improves collaboration**, since teams must think about how models are used from several vantage points;
  3. it **speeds up deployment** with an ecosystem that automates pipeline tasks and keeps projects reproducible;
  4. it **enhances monitoring, observability and explainability**, to catch issues such as model drift or technical debt.
  Its closing remark is worth keeping: "there is no such thing as a model that works well forever" — hence monitoring and retraining.
* **Forward reference.** The book says it will later demonstrate pipelines that combine a `ColumnTransformer()` with an estimator such as
  `RandomForestClassifier()` to unify preprocessing, model selection and cross-validation in one object (it cites a later chapter by number; `ColumnTransformer` is introduced in Chapter 2 of this repository's notebooks).
* **Security note:** `pickle`/`joblib` files can execute code when loaded — only load files you trust.

## 6. Common attributes and methods

**Summary.** Fitted models expose what they learned through attributes ending with an underscore (`coef_`, `intercept_`,
`classes_`, `feature_importances_`…) and a `score()` method that returns a default metric — **accuracy** for classifiers,
**R²** for regressors.

### Reproduction

In [56]:
from sklearn.linear_model import LinearRegression
import numpy as np

# Example data
X = np.array([[1], [2], [3], [4], [5]])  # Feature matrix
y = np.array([1, 2, 3, 3.5, 5])  # Target values

# Create and fit the model
model = LinearRegression()
model.fit(X, y)

# Access coefficients (slope of the linear model)
print("Coefficients:", model.coef_)

# Access y-intercept
print("Intercept:", model.intercept_)

# Use score() method to evaluate the model (R-squared value)
print("Model R-squared:", model.score(X, y))

Coefficients: [0.95]
Intercept: 0.04999999999999938
Model R-squared: 0.9809782608695652


**Verify R² by hand** and compute the *adjusted* R² the book's note mentions:

In [57]:
y_hat = model.predict(X)
ss_res = ((y - y_hat) ** 2).sum()
ss_tot = ((y - y.mean()) ** 2).sum()
r2 = 1 - ss_res / ss_tot
n, p = X.shape
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
print("R2 manual   :", r2)
print("Adjusted R2 :", adj_r2)

R2 manual   : 0.9809782608695652
Adjusted R2 : 0.9746376811594203


### Theory
* **Trailing underscore convention:** attributes *learned from data* end with `_`; constructor hyperparameters do not.
  They exist only after `fit()`.
* **`score()` defaults:** classifiers → mean accuracy; regressors → coefficient of determination
  $R^2 = 1-\dfrac{SS_{res}}{SS_{tot}}$ (1 = perfect, 0 = no better than predicting the mean, can be negative).
* **R² caveat (book's note):** R² never decreases when features are added, even useless ones. **Adjusted R²**
  $=1-(1-R^2)\dfrac{n-1}{n-p-1}$ penalises the number of predictors $p$ ($n$ = samples).
* Here the score is computed on the *training* data (as in the book) — it is optimistic. Use a held-out set or
  cross-validation for honest estimates (Chapter 12).

## 7. Hyperparameter tuning with search methods

**Summary.** *Hyperparameters* are settings you choose before training (e.g. `n_estimators`, `max_depth`).
`get_params()` reads them and `set_params()` changes them. For automatic search, scikit-learn provides
`GridSearchCV` (try every combination), `RandomizedSearchCV` (sample combinations) and the *successive halving*
variants `HalvingGridSearchCV` / `HalvingRandomSearchCV`.

### Reproduction — `set_params()` / `get_params()`

In [58]:
from sklearn.ensemble import RandomForestClassifier

# Create a RandomForestClassifier model
model = RandomForestClassifier()

# Set hyperparameters prior to training using set_params()
model.set_params(n_estimators=100, max_depth=10, random_state=42)

# Check the updated parameters
print(model.get_params())

{'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': 10, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100, 'n_jobs': None, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}


> The printed dictionary depends on the installed version (e.g. newer versions add `monotonic_cst`, the book's output
> includes it too). The values we set — `n_estimators=100`, `max_depth=10`, `random_state=42` — are what matter.

**Extension — the searches themselves** (the book names them but shows no code).

In [59]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.svm import SVC

pipe_svc = Pipeline([("scaler", StandardScaler()), ("svc", SVC())])
param_grid = {"svc__C": [0.1, 1, 10], "svc__gamma": ["scale", 0.01, 0.1]}   # stepname__param

grid = GridSearchCV(pipe_svc, param_grid, cv=5)
grid.fit(X_tr, y_tr)
print("GridSearchCV  best params:", grid.best_params_, "| CV acc:", round(grid.best_score_, 3),
      "| test acc:", round(grid.score(X_te, y_te), 3))
print("combinations tried:", len(grid.cv_results_["params"]), "x 5 folds")

from scipy.stats import loguniform
rand = RandomizedSearchCV(pipe_svc,
                          {"svc__C": loguniform(1e-2, 1e2), "svc__gamma": loguniform(1e-3, 1)},
                          n_iter=10, cv=5, random_state=42)
rand.fit(X_tr, y_tr)
print("RandomizedSearchCV best params:", {k: round(float(v), 4) for k, v in rand.best_params_.items()},
      "| CV acc:", round(rand.best_score_, 3), "| test acc:", round(rand.score(X_te, y_te), 3))

GridSearchCV  best params: {'svc__C': 1, 'svc__gamma': 0.1} | CV acc: 0.981 | test acc: 0.911
combinations tried: 9 x 5 folds
RandomizedSearchCV best params: {'svc__C': 8.4718, 'svc__gamma': 0.0625} | CV acc: 0.971 | test acc: 0.933


In [60]:
# Successive halving is still flagged experimental and must be enabled explicitly
from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.model_selection import HalvingGridSearchCV

halv = HalvingGridSearchCV(pipe_svc, param_grid, cv=5, random_state=42)
halv.fit(X_tr, y_tr)
print("HalvingGridSearchCV best params:", halv.best_params_, "| test acc:", round(halv.score(X_te, y_te), 3))

HalvingGridSearchCV best params: {'svc__C': 10, 'svc__gamma': 0.01} | test acc: 0.911


### Theory
* **Parameters vs hyperparameters.** *Parameters* (`coef_`, tree splits) are learned by `fit`. *Hyperparameters* control
  the learning process and are chosen by the user, ideally by validation performance — never by test performance.
* **Grid search** is exhaustive: cost = (#combinations) × (#folds) fits. Simple but explodes with many parameters.
* **Random search** (Bergstra & Bengio, 2012) samples a fixed budget `n_iter` from distributions. It is usually more
  efficient when only a few hyperparameters really matter, and handles continuous ranges (`loguniform`).
* **Successive halving** starts all candidates with a *small* resource (few samples), keeps the best fraction
  (`factor`, default 3), gives survivors more resource, and repeats — a tournament that saves compute.
* **`stepname__param`** double-underscore syntax lets a search reach into pipeline steps. Because the whole pipeline
  is refit per fold, preprocessing stays leakage-free.
* The best configuration found on CV is then refit on the whole training set (`refit=True`) and available as
  `best_estimator_`.

## 8. Working with metadata: Tags and more

**Summary.** Two different things are called "metadata" in this section:
1. **Estimator tags** — static facts an estimator declares about itself (does it accept NaNs? is it a classifier? does it
   need positive data?). scikit-learn and meta-estimators read tags to decide behaviour.
2. **Metadata routing** — passing *extra per-sample information* (e.g. `sample_weight`, `groups`) through meta-estimators
   (Pipeline, GridSearchCV, cross-validation) to exactly those steps that *consume* it. A **router** forwards metadata; a
   **consumer** uses it.

The book describes both but gives no code and defers examples to Chapters 12–13.

**Extension — look at tags:**

In [61]:
from sklearn.utils import get_tags
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

for est in (LogisticRegression(), HistGradientBoostingClassifier()):
    t = get_tags(est)
    print(f"{type(est).__name__:32s} estimator_type={t.estimator_type!s:12s} allow_nan={t.input_tags.allow_nan}  sparse={t.input_tags.sparse}")

LogisticRegression               estimator_type=classifier   allow_nan=False  sparse=True
HistGradientBoostingClassifier   estimator_type=classifier   allow_nan=True  sparse=False


`HistGradientBoostingClassifier` declares `allow_nan=True`: it handles missing values natively, so a pipeline does not need an imputer in front of it.

**Extension — metadata routing.** Here `sample_weight` is *requested* by the classifier and *not* by the scaler, so
a pipeline routes it only where it is needed.

In [62]:
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

sklearn.set_config(enable_metadata_routing=True)       # feature is opt-in

weights = np.where(y_tr == 0, 5.0, 1.0)                # up-weight class 0 five times

routed = make_pipeline(
    StandardScaler().set_fit_request(sample_weight=False),                       # scaler: do NOT use weights
    LogisticRegression(max_iter=200).set_fit_request(sample_weight=True),        # classifier: use weights
)
routed.fit(X_tr, y_tr, sample_weight=weights)
print("Fitted with routed sample_weight. Test accuracy:", round(routed.score(X_te, y_te), 3))

sklearn.set_config(enable_metadata_routing=False)      # restore default

Fitted with routed sample_weight. Test accuracy: 0.911


### Theory
* **Tags** make *capabilities* machine-readable. Examples: `allow_nan`, `sparse` (accepts sparse input), `multi_output`,
  `requires_fit`, `estimator_type`. Since scikit-learn 1.6 tags are a dataclass (`get_tags()`); custom estimators
  override `__sklearn_tags__()`. Test helpers (`check_estimator`) use tags to decide which API checks to run.
* **Why metadata routing?** Before it existed, passing `sample_weight` through a `Pipeline` was clumsy
  (`pipe.fit(X, y, clf__sample_weight=w)`) and unsupported for many meta-estimators. Routing makes data flow explicit:
  each consumer declares what it *requests* (`set_fit_request`, `set_score_request`…), each router forwards it.
* The book's imbalanced-data example (weights for the classifier but not the scaler) is exactly the pattern above.

## 9. Best practices for API usage

**Summary (the book's key takeaways).**
* **Uniform API** — `fit`, `transform`, `predict` everywhere → readable, maintainable code.
* **Data preprocessing** — scale/encode/impute with `sklearn.preprocessing` / `impute` *before* modelling.
* **Pipelines** — chain steps to simplify code and tuning.
* **Cross-validation** — estimate generalisation with `sklearn.model_selection`.
* **Hyperparameter tuning** — `GridSearchCV` / `RandomizedSearchCV`.

**Putting it all together (Extension):** one short, leakage-safe, tuned workflow using every best practice above.

In [63]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold

X_bc, y_bc = load_breast_cancer(return_X_y=True)
X_trb, X_teb, y_trb, y_teb = train_test_split(X_bc, y_bc, test_size=0.25, random_state=0, stratify=y_bc)

best_practice = Pipeline([("scale", StandardScaler()), ("clf", LogisticRegression(max_iter=1000))])   # pipeline + preprocessing
search = GridSearchCV(best_practice, {"clf__C": [0.01, 0.1, 1, 10]},                                  # tuning
                      cv=StratifiedKFold(5, shuffle=True, random_state=0))                            # cross-validation
search.fit(X_trb, y_trb)
print("Best C:", search.best_params_["clf__C"])
print("Cross-validated accuracy:", round(search.best_score_, 3))
print("Held-out test accuracy  :", round(search.score(X_teb, y_teb), 3))

Best C: 1
Cross-validated accuracy: 0.981
Held-out test accuracy  : 0.958


### Chapter takeaways
1. Every object follows the same small set of verbs, so learning one algorithm teaches you the API for all of them.
2. *Estimators learn, transformers change data*; learned state lives in `attribute_`; never fit on test data.
3. Pipelines + cross-validation + hyperparameter search form the standard, leak-free workflow used for the rest of the book.
4. The library is extensible (custom estimators/transformers, tags, metadata routing) without leaving the API.

*Next: Chapter 2 — Pre-Model Workflow and Data Preprocessing.*